#### -----------------------------------------------------------------------------<br>Copyright (c) 2024, Lucid Vision Labs, Inc.
##### THE  SOFTWARE  IS  PROVIDED  "AS IS",  WITHOUT  WARRANTY  OF  ANY  KIND,<br>EXPRESS  OR  IMPLIED,  INCLUDING  BUT  NOT  LIMITED  TO  THE  WARRANTIES<br>OF  MERCHANTABILITY,  FITNESS  FOR  A  PARTICULAR  PURPOSE  AND<br>NONINFRINGEMENT.  IN  NO  EVENT  SHALL  THE  AUTHORS  OR  COPYRIGHT  HOLDERS<br>BE  LIABLE  FOR  ANY  CLAIM,  DAMAGES  OR  OTHER  LIABILITY,  WHETHER  IN  AN<br>ACTION  OF  CONTRACT,  TORT  OR  OTHERWISE,  ARISING  FROM,  OUT  OF  OR  IN<br>CONNECTION  WITH  THE  SOFTWARE  OR  THE  USE  OR  OTHER  DEALINGS  IN  THE  SOFTWARE.<br>-----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# Warning:
#
# EVS examples support only on windows at the moment
# -----------------------------------------------------------------------------

In [ ]:
import time

from arena_api.system import system
from arena_api.__future__.save import Writer

#### Save: EVS Save

> This example demonstrates how to acquire and save an image using the Event Stream (EVS)format.
> It covers setting up acquisition mode, configuring the camera for EVS,
> and saving the acquired image in the PLY format using the save library.


In [19]:
TAB1 = "  "
TAB2 = "    "
TAB3 = "	 "
FILE_NAME = "Images/Py_EVS_Save_Ply/Py_EVS_Save_Ply.ply"

In [ ]:
"""
This function waits for the user to connect a device before raising an exception
"""

tries = 0
tries_max = 6
sleep_time_secs = 10
while tries < tries_max:  # Wait for device for 60 seconds
    devices = system.create_device()
    if not devices:
        print(
            f'{TAB1}Try {tries+1} of {tries_max}: waiting for {sleep_time_secs} '
            f'secs for a device to be connected!')
        for sec_count in range(sleep_time_secs):
            time.sleep(1)
            print(f'{TAB1}{sec_count + 1 } seconds passed ',
                  '.' * sec_count, end='\r')
            tries += 1
    else:
        print(f'{TAB1}Created {len(devices)} device(s)')
        device = system.select_device(devices)
        nodemap = device.nodemap
        tl_stream_nodemap = device.tl_stream_nodemap
        break
else:
    raise Exception(f'{TAB1}No device found! Please connect a device and run '
                    f'the example again.')


In [ ]:
width = nodemap.get_node("Width").value
height = nodemap.get_node("Height").value
print(f'{TAB1}Image (w, h) = ({width} , {height} )')

##### Configure device settings

In [ ]:
print(f'{TAB1}Set acquisition mode to \'Continuous\'')
initial_acquisition_mode = nodemap.get_node("AcquisitionMode").value
nodemap.get_node("AcquisitionMode").value = "Continuous"

In [ ]:
print(f'{TAB1}Set buffer handling mode to \'NewestOnly\'')
tl_stream_nodemap["StreamBufferHandlingMode"].value = "NewestOnly"

#### Acquisition: EVS
>	The EventFormat node determines whether the camera can use the EVS datastream engine. 
>   When set to EVS, Arena switches to the EVS engine. If EVS is not supported, 
>   the acquisition mode is restored to its original setting, and the process is exited.

In [ ]:
print(f'{TAB1}Set Event Format to EVT3.0')

try:
    event_format_initial = nodemap.get_node('EventFormat').value
    nodemap["EventFormat"].value = "EVT3_0"
except:
    print(f'{TAB1}Connected camera does not support any EventFormats\n')
    nodemap.get_node("AcquisitionMode").value = initial_acquisition_mode
    system.destroy_device()
    raise

##### Set camera event rate to 10 Mev/s

In [ ]:
print(f'{TAB1}Set Camera Event Rate to 10 Mev/s')
erc_enable_initial = nodemap.get_node('ErcEnable').value
nodemap["ErcEnable"].value = True

camera_event_rate_initial = nodemap.get_node('ErcRateLimit').value
nodemap["ErcRateLimit"].value = 10.0

#####  Set evs output format to XYTPFrame

In [ ]:
print(f'{TAB1}Set EVS output format to XYTPFrame')
tl_stream_nodemap["StreamEvsOutputFormat"].value = "XYTPFrame"

##### Image Save Function

 Prepare image parameters and save the image into ply format using writer provide by save library

In [28]:
def save_image(image_buffer, filepath):
	print(f'{TAB2}Prepare image parameters')
	width = image_buffer.width
	height = image_buffer.height
	bits_per_pixel = image_buffer.bits_per_pixel

	'''
	The buffer will be the size of the full image but we need to specify the number of 
	actual vertices stored in the buffer.
	'''
	size_filled = image_buffer.size_filled
	num_vertices = int(size_filled / (bits_per_pixel / 8))

	print(f'{TAB2}Prepare image writer')
	
	writer = Writer(width,height,bits_per_pixel, num_vertices)

	writer.save(image_buffer, filepath)
	print(f'{TAB1}Image saved {writer.saved_images[-1]}')

In [ ]:
print(f'{TAB1}Start stream')
device.start_stream(1)

print(f'{TAB1}Get one image')

buffer = device.get_buffer()


'''
Print image buffer info
    Buffers contain image data.
    Image data can also be copied and converted using BufferFactory.
    That is necessary to retain image data, as we must also requeue the buffer.
'''

if buffer.is_incomplete:
	print(f'{TAB3}Image {buffer.frame_id} is incomplete')
else:
	save_image(buffer,FILE_NAME)
	
device.requeue_buffer(buffer)
print(f'{TAB1}Image buffer requeued')

device.stop_stream()
print(f'{TAB1}Stream stopped')

In [32]:
nodemap.get_node("ErcEnable").value = erc_enable_initial
nodemap.get_node("ErcRateLimit").value = camera_event_rate_initial
nodemap.get_node("EventFormat").value = event_format_initial

##### Clean up ----------------------------------------------------------------

> - Destroy device. This call is optional and will automatically be
  called for any remaining devices when the system module is unloading.

In [ ]:
nodemap.get_node("AcquisitionMode").value = initial_acquisition_mode

system.destroy_device()
print('Destroyed all created devices')